# 02_nettoyage_openchargemap.ipynb

In [1]:
import json
import polars as pl
from datetime import datetime
from pathlib import Path
ROOT_PATH = Path.cwd().resolve().parent

path_target_file = ROOT_PATH / "data" / "raw" / "2026-07-21_132047_paris_extract.json"

with open(path_target_file, "r") as f:
    fichier = json.load(f)

fichier_list = [
                    {    
                    "poi_id"              : poi["ID"],
                    "title"               : poi["AddressInfo"]["Title"],
                    "town"                : poi["AddressInfo"]["Town"],
                    "postcode"            : poi["AddressInfo"]["Postcode"],
                    "latitude"            : poi["AddressInfo"]["Latitude"],
                    "longitude"           : poi["AddressInfo"]["Longitude"],
                    "number_of_points"    : poi["NumberOfPoints"],
                    "usage_cost"          : poi["UsageCost"],
                    "date_last_confirmed" : poi["DateLastConfirmed"]
                    } 
                    for poi in fichier
            ]
fichier_list

In [2]:
table_poi = []
for poi in fichier:
    poi_dict = {}
    poi_dict["poi_id"]              = poi["ID"]
    poi_dict["title"]               = poi["AddressInfo"]["Title"]
    poi_dict["town"]                = poi["AddressInfo"]["Town"]
    poi_dict["postcode"]            = poi["AddressInfo"]["Postcode"]
    poi_dict["latitude"]            = poi["AddressInfo"]["Latitude"]
    poi_dict["longitude"]           = poi["AddressInfo"]["Longitude"]
    poi_dict["number_of_points"]    = poi["NumberOfPoints"]
    poi_dict["usage_cost"]          = poi["UsageCost"]
    poi_dict["date_last_confirmed"] = poi["DateLastConfirmed"]
    table_poi.append(poi_dict)
table_poi

[{'poi_id': 7008,
  'title': 'angle rue Poulet - Barbès',
  'town': 'Paris',
  'postcode': '',
  'latitude': 48.856614,
  'longitude': 2.3522219000000177,
  'number_of_points': 1,
  'usage_cost': 'Free',
  'date_last_confirmed': '2011-10-12T20:41:00Z'},
 {'poi_id': 6931,
  'title': 'Parking Lobau-Rivoli',
  'town': 'Paris',
  'postcode': ' 75004',
  'latitude': 48.8560139,
  'longitude': 2.353280600000062,
  'number_of_points': 1,
  'usage_cost': 'Free',
  'date_last_confirmed': '2011-10-10T18:18:00Z'},
 {'poi_id': 60368,
  'title': 'Place Saint-Gervais',
  'town': 'Paris',
  'postcode': '75004',
  'latitude': 48.855837,
  'longitude': 2.35409470000002,
  'number_of_points': 3,
  'usage_cost': '>3kW: 0,25€/15min <60min; 2,00€/15min >60min <75min; 4,00€/15min >=75min. Au-delà de la première heure, 2€ les 15 premières minutes supplémentaires puis 4€ /quart d’heure',
  'date_last_confirmed': None},
 {'poi_id': 6977,
  'title': 'Place Saint-Gervais autolib',
  'town': 'Paris',
  'postcode'

In [3]:
table_connections =[]
for poi in fichier:
    for connection in poi["Connections"]:
        connection_dict = {}
        connection_dict["poi_id"]  = poi["ID"]
        connection_dict["connection_id"] = connection["ID"]
        connection_dict["power_kw"] = connection["PowerKW"]
        connection_dict["amps"] = connection["Amps"]
        connection_dict["voltage"] = connection["Voltage"]
        connection_dict["connection_type"] = connection["ConnectionType"]["Title"]
        connection_dict["current_type"] = connection["CurrentType"]["Title"]
        connection_dict["is_operational"] = connection["StatusType"]["IsOperational"]
        connection_dict["level_title"] = connection["Level"]["Title"]
        connection_dict["is_fast_charge_capable"] = connection["Level"]["IsFastChargeCapable"]
        table_connections.append(connection_dict)
table_connections


[{'poi_id': 60368,
  'connection_id': 76450,
  'power_kw': 22,
  'amps': 120,
  'voltage': 400,
  'connection_type': 'CCS (Type 2)',
  'current_type': 'DC',
  'is_operational': True,
  'level_title': 'Level 3:  High (Over 40kW)',
  'is_fast_charge_capable': True},
 {'poi_id': 60368,
  'connection_id': 76451,
  'power_kw': 22,
  'amps': 120,
  'voltage': 400,
  'connection_type': 'CHAdeMO',
  'current_type': 'DC',
  'is_operational': True,
  'level_title': 'Level 3:  High (Over 40kW)',
  'is_fast_charge_capable': True},
 {'poi_id': 60368,
  'connection_id': 76452,
  'power_kw': 22,
  'amps': 32,
  'voltage': 400,
  'connection_type': 'Type 2 (Socket Only)',
  'current_type': 'AC (Three-Phase)',
  'is_operational': True,
  'level_title': 'Level 2 : Medium (Over 2kW)',
  'is_fast_charge_capable': False},
 {'poi_id': 60368,
  'connection_id': 76453,
  'power_kw': 22,
  'amps': 32,
  'voltage': 400,
  'connection_type': 'SCAME Type 3C (Schneider-Legrand)',
  'current_type': 'AC (Three-Phase

In [4]:
names = [
    ("paris_poi", table_poi),
    ("connections", table_connections)
]
now = datetime.now().strftime("%Y-%m-%d_%H%M%S")
for name, data in names:
    df = pl.DataFrame(data)
    path_target_file = ROOT_PATH / "data" / "processed" / f"{now}_{name}.parquet"
    df.write_parquet(path_target_file)